# Initial Exploration
This notebook is used to get to know the dataset. 

Explore structure, understand columns, data types, missing values, incosistencies, etc.

In [1]:
import pandas as pd

df = pd.read_csv("../data/naloxone.csv")

In [3]:
df

,ID,Incident Number,Dispatch Date,Patient Number,Age,Gender,Ward,Neighbourhood ID,Neighbourhood,Naxolone Administrations
0,2021095567##1,2021095567,2021-08-08T04:33:22,1,30 to 34,Male,NaN,NaN,NaN,1
1,2021095606##1,2021095606,2021-08-08T07:41:30,1,55 to 59,Female,Fort Rouge - East Fort Garry,NE168,South Portage,1
2,2021095828##1,2021095828,2021-08-08T16:43:40,1,50 to 54,Female,Fort Rouge - East Fort Garry,NE015,Broadway-Assiniboine,1
3,2021095933##1,2021095933,2021-08-08T20:43:56,1,35 to 39,Female,NaN,NaN,NaN,2
4,2021095940##1,2021095940,2021-08-08T21:08:20,1,25 to 29,Male,Point Douglas,NE219,West Alexander,1
...,...,...,...,...,...,...,...,...,...,...
26317,2026032828##1,2026032828,2026-03-04T15:19:09,1,25 to 29,Male,Point Douglas,NE166,South Point Douglas,1
26318,2026032867##1,2026032867,2026-03-04T16:45:56,1,30 to 34,Male,Point Douglas,NE126,Omand's Creek Industrial,3
26319,2026033071##1,2026033071,2026-03-05T00:01:28,1,55 to 59,Male,Daniel McIntyre,NE025,Central Park,3
26320,2026033477##1,2026033477,2026-03-05T18:49:53,1,30 to 34,Female,Mynarski,NE118,North Point Douglas,1


## General Structure

In [4]:
# general structure
print("Shape:", df.shape)
df.info()

Shape: (26322, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26322 entries, 0 to 26321
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   ID                        26322 non-null  object
 1   Incident Number           26322 non-null  int64 
 2   Dispatch Date             26322 non-null  object
 3   Patient Number            26322 non-null  int64 
 4   Age                       26189 non-null  object
 5   Gender                    26322 non-null  object
 6   Ward                      25984 non-null  object
 7   Neighbourhood ID          25984 non-null  object
 8   Neighbourhood             25984 non-null  object
 9   Naxolone Administrations  26322 non-null  int64 
dtypes: int64(3), object(7)
memory usage: 2.0+ MB


- The dataset has 10 columns. Some of them I don't think will bring much analytical value, like `ID`, `Incident Number`, `Neighbourhood ID`.
- Around 26k rows -> I wonder the range of years that it covers.
- Consistent column naming, however, I prefer snake_case.
- Typo in column name, `NaXOlone Administrations`. The correct spelling is `NaLOxone Administrations`.
- Dataset seems to be ordered by `Dispatch Date` (looking at the 5 first and last rows).

## Basic Statistics (numerical columns)

In [5]:
df.describe()

,Incident Number,Patient Number,Naxolone Administrations
count,2.632200e+04,26322.000000,26322.000000
mean,2.018549e+09,1.433326,1.579553
std,5.442832e+07,0.944280,1.017530
min,1.000000e+00,1.000000,1.000000
25%,2.017063e+09,1.000000,1.000000
50%,2.021044e+09,1.000000,1.000000
75%,2.023153e+09,2.000000,2.000000
max,2.026047e+09,14.000000,12.000000


-  4 first digits of `Incident Number` seems to correspond to the year the incident happened.
    - I see the 25% starting with 2017. Maybe the dataset is not ordered then.
- `Patient Number` it's very concentrated near 1, but with a maximum value of 14.
- `Naxolone Administrations`: Just as `Patient Number`, it's very clustered near in 1.
    - Mean of 1.5 with a max of 12 -> extreme outlier (someone seems to took 12 doses of Naloxone).


## Checking Missing Values

In [6]:
df.isna().sum()

ID                            0
Incident Number               0
Dispatch Date                 0
Patient Number                0
Age                         133
Gender                        0
Ward                        338
Neighbourhood ID            338
Neighbourhood               338
Naxolone Administrations      0
dtype: int64

- Not that much missing values
- Columns `Ward`, `Neighbourhood ID`, and `Neighbourhood` have exactly the same number of missing values, probably meaning that location is missing.

## Analyzing Specific Columns

### `Dispatch Date`

In [8]:
# checking the type of values (more specifically than "object")
type(df["Dispatch Date"][0])

str

In [9]:
# trying to get the range of dates the dataset cover
print(df["Dispatch Date"].min())
print(df["Dispatch Date"].max())

2007-11-21T14:01:14
2026-03-30T22:14:46


- The data type is "wrong" here. It's being interpreted as pure text.
- Keeps date and timestamp together
- It looks like the range of dates the dataset covers is 2007-2026.
- This tells me that the dataset is not ordered by `Dispatch Date`, it was just a coincidence.

### `Patient Number`

In [7]:
df["Patient Number"].value_counts()

Patient Number
1     19687
2      3937
3      1522
4       676
5       289
6       115
7        50
8        25
9        10
10        5
11        3
12        1
13        1
14        1
Name: count, dtype: int64

From what I understood, this column holds the patient number **of that specifc incident**. Meaning that one incident can have more than one patient.

- Bulk of incidents are single person.
- Strong right-skew with some outliers of incidents with more than 5 persons involved.

### `Age`

In [10]:
df["Age"].value_counts()

Age
30 to 34    4741
25 to 29    4567
35 to 39    3944
20 to 24    3098
40 to 44    2621
45 to 49    1880
50 to 54    1428
15 to 19    1140
55 to 59    1060
60 to 64     491
65 to 69     308
70 to 74     157
Unknown      151
75 to 79     140
10 to 14     117
80 to 84     113
85 to 89     104
90 to 94      68
0 to 4        26
95 to 99      18
Over 100      12
5 to 9         5
Name: count, dtype: int64

- Age is being stored as text and within ranges with distance of 5 years.
- Besides the real missing values (empty strings - NaN) the column has "Unknown" as a value, probably meaning the same thing.

### `Gender`

In [11]:
df["Gender"].value_counts(normalize=True)

Gender
Male       0.597713
Female     0.394575
Unknown    0.007712
Name: proportion, dtype: float64

- The same "problem" happens here: the column has "Unknown" values instead of real missing values.
    - It's probably a thing I'll have to clean in all columns.
- There are more male incidents in the dataset, however the imbalance is not strong (60%-40%)

### `Ward`

In [12]:
df["Ward"].value_counts()

Ward
Point Douglas                      5813
Daniel McIntyre                    5282
Mynarski                           5270
Fort Rouge - East Fort Garry       3255
St. James                          1177
Elmwood - East Kildonan            1051
St. Boniface                        750
St. Vital                           598
Old Kildonan                        499
North Kildonan                      499
Transcona                           449
River Heights - Fort Garry          422
Charleswood - Tuxedo - Westwood     389
St. Norbert - Seine River           305
Waverley West                       225
Name: count, dtype: int64

- Strong concentration of incidents in 3 regions.
- Consistent values, not much to point out here.

### `Neighbourhood`

In [13]:
df["Neighbourhood"].value_counts()

Neighbourhood
William Whyte               1758
South Point Douglas         1410
Spence                      1241
South Portage               1097
Central Park                1069
                            ... 
Bridgwater Centre              1
Kingston Crescent              1
Woodhaven                      1
West Kildonan Industrial       1
Agassiz                        1
Name: count, Length: 223, dtype: int64

- Tons of different neighbourhoods, probably a lot of them have unique cases.
- `Ward` seems to group them in a reasonable number of categories, being a better column to work with.

### `Naxolone Administrations` (typo)


In [14]:
df["Naxolone Administrations"].value_counts()

Naxolone Administrations
1     17227
2      5457
3      2131
4       899
5       375
6       138
7        52
8        25
9         7
11        5
10        5
12        1
Name: count, dtype: int64

- No "Unknown"s here.
- Clear right skew, huge percentage of the incidents are single person.
- Numbers of incidents drops extremely after 4+ administrations.

## Basic Initial Observations Summary
- Dataset has 26k rows and 10 columns.
- Some columns probably won't bring much analytical value: `ID`, `Incident Number`, `Neighbourhood ID`.
- Column naming is consistent, but I prefer snake_case (change that).
- Typo in column name `NaXOlone Administrations`. Correct spelling is `NaLOxone`.
- Not many missing values overall.
- Columns `Ward`, `Neighbourhood ID`, and `Neighbourhood` share exactly the same number of missing values (338), probably just meaning that location is missing.
- `Dispatch Date` is stored as plain text with date and timestamp together. Needs parsing before temporal analysis.
- Dataset covers from 2007 to 2026
- `Patient Number` is very concentrated at 1, but with a right skew with some outliers.
- `Age` uses string ranges (ex: "30 to 34") instead of numerical values, and has two types of missing values: real NaN and filled "Unknown".
- `Gender` has the same "Unknown" problem. Probably something to standardize across all columns.
- `Ward` has a strong concentration in 3 regions.
- `Neighbourhood` has 223 unique values with many having only 1 case.
- `Naxolone Administrations` has a clear right skew with a mean of around 1.5 and a max of 12 (some outliers).


## Analytical Questions That Came to Mind

- How the number of incidents evolved along the years in Winnipeg?

- Is there a temporal pattern in incidents? Some hours of the day, days of week, etc...

- What wards are more affected?

- Is age or gender related to the necessity of multiple administrations?